# Module 7: Probability, the maths of chance

**Goal:** build intuition for randomness, first by **simulating it**, then by checking it with a little maths.
Probability is the bridge from "what happened in my data" to "what would happen next time" (Module 8).

**You will learn:** simulation with NumPy · probability rules · counting · conditional probability and Bayes ·
expected value · binomial and normal distributions · the law of large numbers · the central limit theorem

**Maths in this module:** $P(A)$, complement, addition and multiplication rules, $\binom{n}{k}$, Bayes' theorem,
$E[X]$ and $\operatorname{Var}(X)$, the normal density

> 🎲 Strategy for every problem: **(1) simulate it, (2) work it out with maths, (3) check they agree.**
> Simulation is slower but hard to get wrong, so use it to test your formulas.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", palette="colorblind")
rng = np.random.default_rng(2024)

## 1. Probability as long-run frequency

$P(\text{heads})=0.5$ doesn't mean the next 10 flips give exactly 5 heads. It means that over *many* flips, the **fraction** of heads settles near 0.5.
This is the **law of large numbers** (LLN).

In [ ]:
flips = rng.integers(0, 2, size=10_000)                  # 1 = heads
running_share = np.cumsum(flips) / np.arange(1, len(flips) + 1)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(running_share)
ax.axhline(0.5, color="tab:red", linestyle="--")
ax.set_xscale("log")
ax.set_xlabel("Number of flips (log scale)")
ax.set_ylabel("Share of heads")
ax.set_title("Law of large numbers: the running share settles at 0.5")
plt.show()
print("after 10 flips:", running_share[9], "| after 10,000:", running_share[-1])

## 2. The rules of probability

* An **event** $A$ is a set of outcomes; $0\le P(A)\le1$. The probabilities of all outcomes add to 1.
* **Complement:** $P(\text{not }A)=1-P(A)$
* **Addition (either one):** $P(A\text{ or }B)=P(A)+P(B)-P(A\text{ and }B)$
* **Multiplication (both), independent events:** $P(A\text{ and }B)=P(A)\,P(B)$

### The "at least one" trick
*What is the chance of at least one six in 4 rolls of a die?* Counting all the ways is painful. The complement is easy:
"no sixes" means every roll is a non-six, so

$$P(\text{at least one six})=1-\left(\tfrac56\right)^4$$

In [ ]:
exact = 1 - (5 / 6) ** 4

rolls = rng.integers(1, 7, size=(100_000, 4))             # 100,000 experiments of 4 rolls each
simulated = (rolls == 6).any(axis=1).mean()

print(f"exact {exact:.4f}   simulated {simulated:.4f}")

## 3. Counting and the birthday problem

The number of ways to choose $k$ items from $n$ (order doesn't matter) is $\binom{n}{k}=\dfrac{n!}{k!(n-k)!}$:

In [ ]:
print(math.comb(5, 2), "ways to pick 2 people from 5")
print(math.comb(52, 5), "possible 5-card poker hands")

**Birthday problem:** in a room of 23 people, how likely is it that **at least two share a birthday**?
Most people guess about 6%. Use the complement: people arrive one at a time; each must avoid all earlier birthdays:

$$P(\text{no match})=\frac{365}{365}\cdot\frac{364}{365}\cdots\frac{365-n+1}{365}$$

In [ ]:
def birthday_exact(n):
    return 1 - math.prod((365 - i) / 365 for i in range(n))

def birthday_sim(n, trials=20_000):
    days = rng.integers(0, 365, size=(trials, n))
    has_match = [len(set(row)) < n for row in days]
    return np.mean(has_match)

for n in [10, 23, 40]:
    print(f"n={n:2d}  exact {birthday_exact(n):.3f}   simulated {birthday_sim(n):.3f}")

About 50% with just 23 people! Human intuition for chance is poor, which is exactly why we calculate.

## 4. Conditional probability and Bayes' theorem

$P(A\mid B)$ is the probability of $A$ **given that** $B$ happened:

$$P(A\mid B)=\frac{P(A\text{ and }B)}{P(B)}\qquad\qquad
\textbf{Bayes: }\;P(A\mid B)=\frac{P(B\mid A)\,P(A)}{P(B)}$$

### Example: a medical test
A disease affects **1%** of people. A test detects it **90%** of the time (sensitivity) but also gives a false alarm for **5%** of healthy people.
*If you test positive, what's the chance you have the disease?*

The easiest way to think: imagine **1,000 people** ("natural frequencies").

In [ ]:
people = 1000
sick = people * 0.01                    # 10 people
healthy = people - sick                 # 990 people

true_positives = sick * 0.90            # 9
false_positives = healthy * 0.05        # 49.5

p_sick_given_positive = true_positives / (true_positives + false_positives)
print(f"{true_positives:.0f} true positives vs {false_positives:.1f} false positives")
print(f"P(sick | positive) = {p_sick_given_positive:.3f}")

In [ ]:
# Check by simulation
n = 1_000_000
sick = rng.random(n) < 0.01
positive = np.where(sick, rng.random(n) < 0.90, rng.random(n) < 0.05)
print("simulated P(sick | positive) =", round(sick[positive].mean(), 3))

Only **about 15%**, despite a "90% accurate" test. Because the disease is rare, most positives are false alarms.
This **base-rate** effect appears in spam filters, fraud detection and medical screening, so always ask "how common is the thing I'm looking for?"

## 5. Random variables, expected value and variance

A **random variable** $X$ is a number produced by a random process (the face of a die; a day's rainfall).

$$E[X]=\sum_x x\,P(X=x)\qquad
\operatorname{Var}(X)=E\big[(X-E[X])^2\big]\qquad \text{sd}=\sqrt{\operatorname{Var}(X)}$$

$E[X]$ is the long-run average, and the weights are probabilities.

In [ ]:
faces = np.arange(1, 7)
p = np.full(6, 1 / 6)

ev = (faces * p).sum()
var = ((faces - ev) ** 2 * p).sum()
print("E[die] =", ev, "  Var =", round(var, 3), "  sd =", round(var ** 0.5, 3))

sim = rng.integers(1, 7, size=200_000)
print("simulated mean/var:", sim.mean().round(3), sim.var().round(3))

### Is a game worth playing?
Pay \$3 to roll a die. You win \$6 on a six, \$2 on a 4 or 5, else nothing.

In [ ]:
payoffs = np.array([0, 0, 0, 2, 2, 6])             # for faces 1..6
expected_win = payoffs.mean()
print("expected winnings:", round(expected_win, 3), "-> expected net:", round(expected_win - 3, 3))

plays = rng.choice(payoffs, size=100_000) - 3
print("average net over 100,000 plays:", plays.mean().round(3))

## 6. Distributions

A **distribution** lists the probabilities of each possible value. `scipy.stats` has dozens, each with the same methods:
`pmf`/`pdf` (probability or density), `cdf` (probability of $\le x$), `ppf` (inverse of the cdf), `rvs` (random samples).

### Binomial: count the successes in $n$ independent tries
$$P(X=k)=\binom{n}{k}p^k(1-p)^{n-k},\qquad E[X]=np,\qquad \operatorname{Var}(X)=np(1-p)$$

*Example: 10 coin flips, how likely are exactly 3 heads?*

In [ ]:
k = 3
by_formula = math.comb(10, k) * 0.5 ** k * 0.5 ** (10 - k)
print(by_formula, stats.binom.pmf(k, n=10, p=0.5))

ks = np.arange(0, 11)
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(ks, stats.binom.pmf(ks, 10, 0.5))
ax.set_xlabel("Number of heads in 10 flips"); ax.set_ylabel("Probability")
plt.show()
print("P(at least 8 heads) =", round(1 - stats.binom.cdf(7, 10, 0.5), 4))

### Normal: the bell curve

$$f(x)=\frac{1}{\sigma\sqrt{2\pi}}\exp\!\Big(-\frac{(x-\mu)^2}{2\sigma^2}\Big)$$

It's described by its mean $\mu$ and standard deviation $\sigma$. The **68–95–99.7 rule**: about 68% of values lie within $1\sigma$ of $\mu$, 95% within $2\sigma$, 99.7% within $3\sigma$.

In [ ]:
print([round(stats.norm.cdf(k) - stats.norm.cdf(-k), 4) for k in (1, 2, 3)])
print("z for the 97.5th percentile:", round(stats.norm.ppf(0.975), 3))   # the famous 1.96

In [ ]:
# Setup: find the course data folder (works locally and in Google Colab)
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    if not Path("/content/data-analysis-learnbook").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/PathornKiti/data-analysis-learnbook.git",
                        "/content/data-analysis-learnbook"], check=True)
    DATA = Path("/content/data-analysis-learnbook/data")
else:
    DATA = next(p / "data" for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "data" / "students.csv").exists())
print("Data folder:", DATA)

**Do real data look normal?** Compare the exam scores to a normal curve with the same mean and standard deviation.

In [ ]:
scores = pd.read_csv(DATA / "students.csv")["exam_score"]
mu, sigma = scores.mean(), scores.std()

fig, ax = plt.subplots(figsize=(6.5, 3.5))
ax.hist(scores, bins=20, density=True, alpha=0.6, edgecolor="white", label="data")
xs = np.linspace(scores.min(), scores.max(), 200)
ax.plot(xs, stats.norm.pdf(xs, mu, sigma), color="tab:red", linewidth=2, label=f"Normal({mu:.1f}, {sigma:.1f})")
ax.set_xlabel("Exam score"); ax.set_ylabel("Density"); ax.legend()
plt.show()

p_model = 1 - stats.norm.cdf(80, mu, sigma)
print(f"share scoring 80+: model {p_model:.3f}   actual {(scores >= 80).mean():.3f}")

The model is close, not perfect. Real data are *approximately* normal at best. (Scores are capped at 100, for one thing.)
Every model is a useful simplification; check it against the data.

## 7. The central limit theorem: why the normal distribution is everywhere

Take **averages** of random samples and something remarkable happens, whatever the shape of the original data:

1. the averages follow (approximately) a **normal** distribution,
2. centred on the true mean $\mu$,
3. with spread $\dfrac{\sigma}{\sqrt n}$ (the **standard error**), which shrinks as the sample grows.

Start with a very skewed distribution (exponential, with mean 1 and standard deviation 1):

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3), sharex=True)
axes[0].hist(rng.exponential(1, 20_000), bins=60, density=True)
axes[0].set_title("one value (skewed)")
for ax, n in zip(axes[1:], [2, 10, 50]):
    means = rng.exponential(1, size=(20_000, n)).mean(axis=1)
    ax.hist(means, bins=60, density=True)
    ax.set_title(f"mean of n={n}\nsd ≈ {means.std():.2f} (theory {1 / math.sqrt(n):.2f})")
for ax in axes: ax.set_xlim(0, 4)
plt.tight_layout(); plt.show()

The averages become bell-shaped and narrower. This is the reason confidence intervals and many statistical tests work,
and it's the main idea of the next module.

## Recap

* Probability = long-run frequency (LLN). **Simulate, compute, compare.**
* Complement trick for "at least one" · multiply for independent "and" · add (minus overlap) for "or"
* Bayes: $P(A\mid B)$ depends on the **base rate** $P(A)$. Rare events give mostly false alarms.
* $E[X]$ is the weighted average; Var and sd measure spread
* Binomial counts successes; normal is the bell curve (68–95–99.7)
* CLT: averages are ~normal with sd $\sigma/\sqrt n$

**Next:** `exercises.ipynb`, [`math-notes.md`](math-notes.md), then the project checkpoint in the README.